# ITSELF + Identity sampler + learned reranker on RSTPReid

This notebook trains the upstream ITSELF retrieval model with the Identity sampler, initialized from the attached `rstp.pth`, then trains the project's image-caption token matcher on the train split and evaluates baseline vs reranking on test.

Before running: attach `tbps-benchmark` and the Kaggle model containing `rstp.pth`; enable Internet and a GPU. Training uses `val` for model selection; test is used only in the final evaluation.

The reranker stage uses ITSELF's trained CLIP encoder token features and reranks the global-CLIP top 100. It is a research prototype, not the official ITSELF `global+GRAB` scoring implementation.

In [ ]:
from pathlib import Path
from collections import Counter
import json, subprocess, sys, torch

INPUT_DIR = Path('/kaggle/input')
annotations = list(INPUT_DIR.rglob('data_captions.json'))
clip_checkpoints = list(INPUT_DIR.rglob('rstp.pth'))
if len(annotations) != 1:
    raise RuntimeError(f'Expected one RSTPReid data_captions.json; found {len(annotations)}: {annotations[:5]}')
if len(clip_checkpoints) != 1:
    raise RuntimeError(f'Expected one rstp.pth; found {len(clip_checkpoints)}: {clip_checkpoints[:5]}')
RSTP_ROOT = annotations[0].parent
if not (RSTP_ROOT / 'imgs').is_dir():
    raise FileNotFoundError(f'No imgs/ under {RSTP_ROOT}')
CLIP_CHECKPOINT = clip_checkpoints[0]
records = json.loads(annotations[0].read_text(encoding='utf-8'))
print('RSTP root:', RSTP_ROOT)
print('Checkpoint:', CLIP_CHECKPOINT)
print('Records/splits:', len(records), Counter(r.get('split') for r in records))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU before training.')

## 1. Install this project and the upstream ITSELF implementation

This clones the public TBPS-Sampler repository and installs its lightweight runtime dependencies without reinstalling Kaggle's PyTorch build.

In [ ]:
PROJECT_DIR = Path('/kaggle/working/clip-person-search')
ITSELF_REPO = Path('/kaggle/working/tbps-sampler')
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/lqb464/clip-person-search.git', str(PROJECT_DIR)], check=True)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/HoangVo-Prog/tbps-sampler.git', str(ITSELF_REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(PROJECT_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'easydict', 'prettytable', 'tensorboard', 'wandb', 'pynvml', 'simple-parsing'], check=True)
ITSELF_DIR = ITSELF_REPO / 'itself'
print('Project:', PROJECT_DIR)
print('Upstream ITSELF:', ITSELF_DIR)

## 2. Adapt the attached RSTP CLIP checkpoint for ITSELF initialization

The Kaggle checkpoint may wrap its CLIP weights under `base_model.` and include task-specific classifier heads. This cell unwraps known containers/prefixes; ITSELF's loader will copy only matching CLIP encoder tensors and resize positional embeddings if necessary.

In [ ]:
payload = torch.load(CLIP_CHECKPOINT, map_location='cpu', weights_only=False)
while isinstance(payload, dict):
    nested = next((payload[k] for k in ('state_dict', 'model_state_dict', 'model') if isinstance(payload.get(k), dict) and any(isinstance(v, torch.Tensor) for v in payload[k].values())), None)
    if nested is None:
        break
    payload = nested
clean_state = {}
for key, value in payload.items():
    if not isinstance(value, torch.Tensor):
        continue
    key = key
    changed = True
    while changed:
        changed = False
        for prefix in ('module.', 'model.', 'base_model.', 'clip_model.', 'clip.'):
            if key.startswith(prefix):
                key = key[len(prefix):]
                changed = True
                break
    clean_state[key] = value
if 'visual.conv1.weight' not in clean_state or 'token_embedding.weight' not in clean_state:
    raise ValueError('Could not find CLIP ViT/text encoder weights after unwrapping checkpoint.')
print('CLIP conv1 shape:', tuple(clean_state['visual.conv1.weight'].shape))
print('CLIP image positional embedding:', tuple(clean_state['visual.positional_embedding'].shape))
ITSELF_INIT = Path('/kaggle/working/rstp_clip_for_itself.pth')
torch.save(clean_state, ITSELF_INIT)
print('Saved ITSELF initialization checkpoint:', ITSELF_INIT)

## 3. Train ITSELF with the Identity sampler

This follows the upstream ITSELF training entry point and default `tal+cid` losses. `val_dataset=val` is explicit so checkpoint selection does not inspect test results. The upstream training default is 60 epochs; reduce `EPOCHS` for a short pipeline check, then rerun fully for a reportable result. The default image size is 384×128; do not pass `--img_size` because upstream parses that option as a Python tuple incorrectly.

In [ ]:
ITSELF_OUTPUT = Path('/kaggle/working/itself-rstpreid')
EPOCHS = 60
BATCH_SIZE_ITSELF = 32  # Lower to 16 if CUDA runs out of memory.
train_cmd = [
    sys.executable, 'train.py',
    '--dataset_name', 'RSTPReid',
    '--root_dir', str(RSTP_ROOT.parent),
    '--pretrain_choice', str(ITSELF_INIT),
    '--sampler', 'identity',
    '--num_instance', '2',
    '--batch_size', str(BATCH_SIZE_ITSELF),
    '--test_batch_size', '128',
    '--num_workers', '2',
    '--num_epoch', str(EPOCHS),
    '--val_dataset', 'val',
    '--eval_period', '1',
    '--eval-after-epoch', '1',
    '--output_dir', str(ITSELF_OUTPUT),
]
subprocess.run(train_cmd, cwd=ITSELF_DIR, check=True)
best_candidates = list(ITSELF_OUTPUT.rglob('best.pth'))
if not best_candidates:
    raise FileNotFoundError(f'ITSELF did not produce best.pth under {ITSELF_OUTPUT}')
ITSELF_CHECKPOINT = max(best_candidates, key=lambda p: p.stat().st_mtime)
print('Selected validation checkpoint:', ITSELF_CHECKPOINT)

## 4. Train the image-caption reranker

The CLIP encoder loaded from the ITSELF checkpoint is frozen; only the cross-modal token fusion matcher is trained. Positives are image-caption pairs from the same identity and negatives use a different identity. The validation split selects the reranker checkpoint.

In [ ]:
RERANKER_CHECKPOINT = '/kaggle/working/rstpreid-itself-reranker.pth'
subprocess.run([
    sys.executable, '-m', 'person_search.cli', 'train-reranker-rstpreid',
    '--root', str(RSTP_ROOT),
    '--clip-checkpoint', str(ITSELF_CHECKPOINT),
    '--clip-model', 'ViT-B-16',
    '--image-size', '384', '128',
    '--image-preprocess', 'itself',
    '--output', RERANKER_CHECKPOINT,
    '--batch-size', '16',
    '--epochs', '5',
    '--num-workers', '2',
], check=True)

## 5. Evaluate on test: baseline vs reranker

Reranking is restricted to the baseline top 100. The report includes R@1/5/10, mAP, deltas, and how many queries lost a previously correct hit at each R@K. The saved per-query CSV helps inspect cases where reranking hurts.

In [ ]:
METRICS_PATH = '/kaggle/working/rstpreid-itself-rerank-metrics.json'
PER_QUERY_PATH = '/kaggle/working/rstpreid-itself-rerank-per-query.csv'
subprocess.run([
    sys.executable, '-m', 'person_search.cli', 'evaluate-rstpreid',
    '--root', str(RSTP_ROOT),
    '--clip-checkpoint', str(ITSELF_CHECKPOINT),
    '--clip-model', 'ViT-B-16',
    '--image-size', '384', '128',
    '--image-preprocess', 'itself',
    '--reranker-checkpoint', RERANKER_CHECKPOINT,
    '--rerank-top-m', '100',
    '--split', 'test',
    '--batch-size', '16',
    '--output', METRICS_PATH,
    '--per-query-output', PER_QUERY_PATH,
], check=True)
print(Path(METRICS_PATH).read_text())
print('Per-query details:', PER_QUERY_PATH)

## Output files

Download from Kaggle's `/kaggle/working`: `best.pth` (ITSELF retrieval model), `rstpreid-itself-reranker.pth`, `rstpreid-itself-rerank-metrics.json`, and `rstpreid-itself-rerank-per-query.csv`. Keep the dataset and model weights out of GitHub.